In [1]:
# !pip install pygit2
# !pip install tree-sitter --upgrade
# !pip install tree-sitter-cpp
# !pip install tree-sitter-c
# !pip install fastparquet
# !pip install pyarrow


In [12]:
import os
import re
import shutil
import pygit2
import pandas as pd
import pyarrow.parquet as pq
import tree_sitter_cpp as tscpp
import tree_sitter_c as tsc
from tree_sitter import Language, Parser
from tqdm.auto import tqdm
from pathlib import Path
import torch
import polars as pl
from collections import defaultdict

device = 0 if torch.cuda.is_available() else -1

In [2]:
def commit_id(commit):
    name = commit.committer.name
    email = commit.committer.email
    time = commit.committer.time
    return '_'.join((name, email, str(time)))

# Download repositories

In [3]:
def extract_functions_from_cpp(content):
    """Extract unique functions and their names from a C++ file using Tree-sitter."""
    tree = parser.parse(content.encode('utf-8'))
    root = tree.root_node
    functions = []
    for node in root.children:
        if node.type == 'function_definition':
            func_text = content[node.start_byte:node.end_byte].strip()
#             print(func_text)
            func_data = {'func_text': func_text}
            for c in node.children:
                if c.type.endswith('type'):
                    func_data['func_return_type'] = content[c.start_byte:c.end_byte].strip()
                if c.type.endswith('declarator'):
                    func_call = content[c.start_byte:c.end_byte].strip().strip('*')
                    i = func_call.find('(')
                    func_name, func_param = func_call[:i], func_call[i:]
                    func_data['func_call'] = func_call
                    func_data['func_name'] = func_name
                    func_data['func_param'] = func_param                    
                elif c.type == 'compound_statement':
                    func_data['func_body'] = content[c.start_byte:c.end_byte].strip()
            functions.append(func_data)
            
    return functions


In [4]:
CPP_LANGUAGE = Language(tsc.language())
parser = Parser(CPP_LANGUAGE)

In [35]:
def clone_repo(repo_url, depth=0):
    """Clone a GitHub repository."""
    if os.path.exists(REPO_DIR):
#         shutil.rmtree(REPO_DIR)
        pass
    else:
        pygit2.clone_repository(repo_url, REPO_DIR, depth=depth)

def get_function_changes(repo_path, output_dir):
    """Extract functions and their changes from all commits."""
    repo = pygit2.Repository(repo_path)
    data = []
    commit_count = sum(1 for commit in repo.walk(repo.head.target, pygit2.GIT_SORT_TOPOLOGICAL))
    i = 1
    partition = 0
    previus_commit = {}
    previus_func_body = {}
    
    for commit in tqdm(repo.walk(repo.head.target, pygit2.GIT_SORT_TOPOLOGICAL), total=commit_count, desc='process commits'):
        current_tree = commit.tree
        parent = commit.parents[0] if commit.parents else None
        parent_tree = parent.tree if parent else None
        
        diff = repo.diff(parent_tree, current_tree) if parent_tree else current_tree.diff_to_tree()
        
        for delta in diff.deltas:
            # Filter for .cpp files
            if delta.new_file and (delta.new_file.path.endswith('.c') or delta.new_file.path.endswith('.cpp')):
                # Get change type
                change_type = delta.status  # pygit2.GIT_DELTA_MODIFIED, GIT_DELTA_ADDED, etc.

                # Retrieve the new blob for modified or added files
                if change_type in (pygit2.GIT_DELTA_MODIFIED, pygit2.GIT_DELTA_ADDED):
                    try:
                        blob = repo[delta.new_file.id]
                        content = blob.data.decode('utf-8')

                        functions = extract_functions_from_cpp(content)
                        for func in functions:                            
                            if func['func_call'] in previus_commit:
                                if func['func_body'] == previus_func_body[func['func_call']]:
                                    continue
                                add_data = {'prev_commit': previus_commit[func['func_call']]}
                            else:
                                add_data = {'prev_commit': None}
                            
                            data.append({
                                'entry_id': delta.new_file.id,
                                'func_id': f"{delta.new_file.path} {func['func_call']}",
                                'file_path': delta.new_file.path,
                                'commit': commit_id(commit),
                                'timestamp': commit.commit_time
                            } | func | add_data)
                            
                            previus_commit[func['func_call']] = commit_id(commit)
                            previus_func_body[func['func_call']] = func['func_body']
                    except Exception as e:
                        print(f"Error in commit {commit_id(commit)}: {e}") 
#                         print(func)

        i += 1   
        if i % 15000 == 0:
            print(f'Save {len(data)} rows')
            df = pd.DataFrame(data)
            df['entry_id'] = df['entry_id'].astype(str)
            df.to_parquet(output_dir / f'{REPO_DIR}_functions_partition{partition}.parquet.gzip', compression='gzip')
            partition += 1
            data = []
#             break
    df = pd.DataFrame(data)
    df['entry_id'] = df['entry_id'].astype(str)
    df.to_parquet(output_dir / f'{REPO_DIR}_functions_partition{partition}.parquet.gzip', compression='gzip') 
    
    print('Number of unique functions:', len(previus_commit))

def link_function_versions(df):
    """Link consecutive versions of functions."""
    df['prev_commit'] = df.groupby('func_id')['commit'].shift(1)
    return df



CPP_LANGUAGE = Language(tsc.language())
parser = Parser(CPP_LANGUAGE)

# repo_url = "https://github.com/catboost/catboost"  # Replace with your GitHub repo

# df = link_function_versions(df)
# save_to_parquet(df, "functions.parquet")
# df.to_parquet(f'{REPO_DIR}_functions.parquet', engine='pyarrow', index=False)
# df.to_csv(f'{REPO_DIR}_functions.csv', index=False)




In [36]:
# Main pipeline
# REPO_DIR = "catboost"
REPO_DIR = "catboost"
repo_url = "https://github.com/catboost/catboost"
output_dir = Path('functions_dataset') / REPO_DIR
if not output_dir.exists():
    os.makedirs(output_dir)

# print("Cloning repository...", REPO_DIR)
# clone_repo(repo_url, depth=1000)
print("Processing functions...")
get_function_changes(REPO_DIR, output_dir)
print("Processing complete. Data saved to functions.parquet.")

Processing functions...


process commits:   0%|          | 0/1000 [00:00<?, ?it/s]

Error in commit pg_pg@yandex-team.com_1728632553: 'func_call'
Error in commit shadchin_shadchin@yandex-team.com_1718298498: 'utf-8' codec can't decode byte 0xe4 in position 86: invalid continuation byte
Number of unique functions: 20184
Processing complete. Data saved to functions.parquet.


In [34]:
get_function_changes(REPO_DIR, output_dir)
print("Processing complete. Data saved to functions.parquet.")

process commits:   0%|          | 0/1000 [00:00<?, ?it/s]

Error in commit shadchin_shadchin@yandex-team.com_1734973811: 'func_return_type'
Error in commit shadchin_shadchin@yandex-team.com_1734973811: 'func_return_type'
Error in commit shadchin_shadchin@yandex-team.com_1734973811: 'func_return_type'
Error in commit shadchin_shadchin@yandex-team.com_1734973811: 'func_return_type'
Error in commit shadchin_shadchin@yandex-team.com_1734973811: 'func_return_type'
Error in commit shadchin_shadchin@yandex-team.com_1734973811: 'func_return_type'
Error in commit shadchin_shadchin@yandex-team.com_1734973811: 'func_return_type'
Error in commit shadchin_shadchin@yandex-team.com_1734973811: 'func_return_type'
Error in commit shadchin_shadchin@yandex-team.com_1734973811: 'func_return_type'
Error in commit shadchin_shadchin@yandex-team.com_1734973811: 'func_return_type'
Error in commit shadchin_shadchin@yandex-team.com_1734973811: 'func_return_type'
Error in commit shadchin_shadchin@yandex-team.com_1734973811: 'func_return_type'
Error in commit shadchin_sha


KeyboardInterrupt



In [13]:
clone_repo(repo_url, depth=1000)

In [14]:
repo = pygit2.Repository(REPO_DIR)
data = []
commit_count = sum(1 for commit in repo.walk(repo.head.target, pygit2.GIT_SORT_TOPOLOGICAL))

In [15]:
commit_count

1000

In [37]:
output_dir = Path('functions_dataset') / REPO_DIR
lazy_df = pl.scan_parquet(output_dir)

In [38]:
lazy_df.select(pl.len()).collect()

len
u32
22110


In [39]:
data_df = lazy_df.collect()

In [40]:
hystory_dict = defaultdict(list)
for func_row in tqdm(data_df.iter_rows(named=True)):
    hystory_dict[func_row['func_id']].append((func_row['func_body'], int(func_row['commit'].split('_')[-1])))

0it [00:00, ?it/s]

In [41]:
len(hystory_dict)

20744

In [42]:
for func_call, func_versions in hystory_dict.items():
    if len(func_versions) > 2:
        break

In [43]:
for i, (func_version, commit_time) in enumerate(func_versions[::-1]):
    print(f'Version {i}',commit_time)
    print(func_version)
    print()

Version 0 1726593561
{
    PyObject *inst, *slotstate;
    PyObject *setstate;
    int status = 0;

    /* Stack is ... instance, state.  We want to leave instance at
     * the stack top, possibly mutated via instance.__setstate__(state).
     */
    if (Py_SIZE(self->stack) - 2 < self->stack->fence)
        return Pdata_stack_underflow(st, self->stack);

    PyObject *state;
    PDATA_POP(st, self->stack, state);
    if (state == NULL)
        return -1;

    inst = self->stack->data[Py_SIZE(self->stack) - 1];

    if (_PyObject_LookupAttr(inst, &_Py_ID(__setstate__), &setstate) < 0) {
        Py_DECREF(state);
        return -1;
    }
    if (setstate != NULL) {
        PyObject *result;

        /* The explicit __setstate__ is responsible for everything. */
        result = _Pickle_FastCall(setstate, state);
        Py_DECREF(setstate);
        if (result == NULL)
            return -1;
        Py_DECREF(result);
        return 0;
    }

    /* A default __setstate__.  First see wh

# read data

In [44]:
REPO_DIR = "qemu"
repo_url = "https://github.com/qemu/qemu"
output_dir = Path('functions_dataset') / REPO_DIR
get_function_changes(REPO_DIR, output_dir)
print("Processing complete. Data saved to functions.parquet.")


process commits:   0%|          | 0/117067 [00:00<?, ?it/s]

Error in commit Peter Maydell_peter.maydell@linaro.org_1727367039: 'func_call'
Error in commit David Hildenbrand_david@redhat.com_1727170414: 'func_call'
Error in commit Peter Maydell_peter.maydell@linaro.org_1726240473: 'func_call'
Error in commit Peter Maydell_peter.maydell@linaro.org_1726240473: 'func_call'
Error in commit Peter Maydell_peter.maydell@linaro.org_1726237904: 'func_call'
Error in commit Peter Maydell_peter.maydell@linaro.org_1726237904: 'func_call'
Error in commit Peter Maydell_peter.maydell@linaro.org_1725986934: 'func_call'
Error in commit Markus Armbruster_armbru@redhat.com_1725969736: 'func_call'
Error in commit Richard Henderson_richard.henderson@linaro.org_1721167152: 'func_call'
Error in commit Richard Henderson_richard.henderson@linaro.org_1721167152: 'func_call'
Error in commit Philippe Mathieu-Daudé_philmd@linaro.org_1721153048: 'func_call'
Error in commit Philippe Mathieu-Daudé_philmd@linaro.org_1721153048: 'func_call'
Error in commit Philippe Mathieu-Daudé_

KeyboardInterrupt: 

In [45]:
lazy_df = pl.scan_parquet(output_dir)
lazy_df.select(pl.len()).collect()

len
u32
11032276


In [46]:
hystory_dict = defaultdict(list)
for func_row in tqdm(lazy_df.iter_rows(named=True)):
    hystory_dict[row['func_id']].append(row['func_body'], int(row['commit'].split('_')[-1]))

AttributeError: 'LazyFrame' object has no attribute 'iter_rows'

In [15]:
res

entry_id,commit,func_id,timestamp,func_call,func_name,func_param,func_body,prev_commit
str,str,str,i64,str,str,str,str,str
"""9b923f6eb4828a5e19bf45e14818a9…","""Michael S. Tsirkin_mst@redhat.…","""amdvi_writew(AMDVIState *s, hw…",1730754204,"""amdvi_writew(AMDVIState *s, hw…","""amdvi_writew""","""(AMDVIState *s, hwaddr addr, u…","""{ uint16_t romask = lduw_l…","""Michael S. Tsirkin_mst@redhat.…"
"""9b923f6eb4828a5e19bf45e14818a9…","""Michael S. Tsirkin_mst@redhat.…","""amdvi_writel(AMDVIState *s, hw…",1730754204,"""amdvi_writel(AMDVIState *s, hw…","""amdvi_writel""","""(AMDVIState *s, hwaddr addr, u…","""{ uint32_t romask = ldl_le…","""Michael S. Tsirkin_mst@redhat.…"
"""9b923f6eb4828a5e19bf45e14818a9…","""Michael S. Tsirkin_mst@redhat.…","""amdvi_writeq(AMDVIState *s, hw…",1730754204,"""amdvi_writeq(AMDVIState *s, hw…","""amdvi_writeq""","""(AMDVIState *s, hwaddr addr, u…","""{ uint64_t romask = ldq_le…","""Michael S. Tsirkin_mst@redhat.…"
"""9b923f6eb4828a5e19bf45e14818a9…","""Michael S. Tsirkin_mst@redhat.…","""amdvi_test_mask(AMDVIState *s,…",1730754204,"""amdvi_test_mask(AMDVIState *s,…","""amdvi_test_mask""","""(AMDVIState *s, hwaddr addr, u…","""{ return amdvi_readq(s, ad…","""Michael S. Tsirkin_mst@redhat.…"
"""9b923f6eb4828a5e19bf45e14818a9…","""Michael S. Tsirkin_mst@redhat.…","""amdvi_assign_orq(AMDVIState *s…",1730754204,"""amdvi_assign_orq(AMDVIState *s…","""amdvi_assign_orq""","""(AMDVIState *s, hwaddr addr, u…","""{ amdvi_writeq_raw(s, addr…","""Michael S. Tsirkin_mst@redhat.…"
…,…,…,…,…,…,…,…,…
"""7c7760c57330c2ae6aa5146cc0016f…","""Michael S. Tsirkin_mst@redhat.…","""amdvi_setevent_bits(uint64_t *…",1730754204,"""amdvi_setevent_bits(uint64_t *…","""amdvi_setevent_bits""","""(uint64_t *buffer, uint64_t va…","""{ int index = start / 64, …","""Michael S. Tsirkin_mst@redhat.…"
"""7c7760c57330c2ae6aa5146cc0016f…","""Michael S. Tsirkin_mst@redhat.…","""amdvi_encode_event(uint64_t *e…",1730754204,"""amdvi_encode_event(uint64_t *e…","""amdvi_encode_event""","""(uint64_t *evt, uint16_t devid…","""{ evt[0] = 0; evt[1] =…","""Michael S. Tsirkin_mst@redhat.…"
"""7c7760c57330c2ae6aa5146cc0016f…","""Michael S. Tsirkin_mst@redhat.…","""amdvi_page_fault(AMDVIState *s…",1730754204,"""amdvi_page_fault(AMDVIState *s…","""amdvi_page_fault""","""(AMDVIState *s, uint16_t devid…","""{ uint64_t evt[2]; in…","""Michael S. Tsirkin_mst@redhat.…"


In [23]:
res = lazy_df.filter(pl.col('func_id').eq("amdvi_writel(AMDVIState *s, hwaddr addr, uint32_t val)")).collect()
res

entry_id,commit,func_id,timestamp,func_call,func_name,func_param,func_body,prev_commit
str,str,str,i64,str,str,str,str,str
"""13af7211e11d7eb665a1899ef15a3c…","""Peter Maydell_peter.maydell@li…","""amdvi_writel(AMDVIState *s, hw…",1730821672,"""amdvi_writel(AMDVIState *s, hw…","""amdvi_writel""","""(AMDVIState *s, hwaddr addr, u…","""{ uint32_t romask = ldl_le…",null
"""13af7211e11d7eb665a1899ef15a3c…","""Michael S. Tsirkin_mst@redhat.…","""amdvi_writel(AMDVIState *s, hw…",1730754204,"""amdvi_writel(AMDVIState *s, hw…","""amdvi_writel""","""(AMDVIState *s, hwaddr addr, u…","""{ uint32_t romask = ldl_le…","""Peter Maydell_peter.maydell@li…"
"""38297376e7a9073d4f226ba6d6343f…","""Michael S. Tsirkin_mst@redhat.…","""amdvi_writel(AMDVIState *s, hw…",1730754204,"""amdvi_writel(AMDVIState *s, hw…","""amdvi_writel""","""(AMDVIState *s, hwaddr addr, u…","""{ uint32_t romask = ldl_le…","""Michael S. Tsirkin_mst@redhat.…"
"""9b923f6eb4828a5e19bf45e14818a9…","""Michael S. Tsirkin_mst@redhat.…","""amdvi_writel(AMDVIState *s, hw…",1730754204,"""amdvi_writel(AMDVIState *s, hw…","""amdvi_writel""","""(AMDVIState *s, hwaddr addr, u…","""{ uint32_t romask = ldl_le…","""Michael S. Tsirkin_mst@redhat.…"
"""7c7760c57330c2ae6aa5146cc0016f…","""Michael S. Tsirkin_mst@redhat.…","""amdvi_writel(AMDVIState *s, hw…",1730754204,"""amdvi_writel(AMDVIState *s, hw…","""amdvi_writel""","""(AMDVIState *s, hwaddr addr, u…","""{ uint32_t romask = ldl_le…","""Michael S. Tsirkin_mst@redhat.…"
…,…,…,…,…,…,…,…,…
"""023de526f6ed16ee6ba2952a182476…","""Paolo Bonzini_pbonzini@redhat.…","""amdvi_writel(AMDVIState *s, hw…",1474970248,"""amdvi_writel(AMDVIState *s, hw…","""amdvi_writel""","""(AMDVIState *s, hwaddr addr, u…","""{ uint32_t romask = ldl_le…","""Peter Maydell_peter.maydell@li…"
"""a868539fb01a8cef7ab32407e68216…","""Paolo Bonzini_pbonzini@redhat.…","""amdvi_writel(AMDVIState *s, hw…",1474959604,"""amdvi_writel(AMDVIState *s, hw…","""amdvi_writel""","""(AMDVIState *s, hwaddr addr, u…","""{ uint32_t romask = ldl_le…","""Paolo Bonzini_pbonzini@redhat.…"
"""a91a1798cb034a383e982909049c89…","""Peter Maydell_peter.maydell@li…","""amdvi_writel(AMDVIState *s, hw…",1474915620,"""amdvi_writel(AMDVIState *s, hw…","""amdvi_writel""","""(AMDVIState *s, hwaddr addr, u…","""{ uint32_t romask = ldl_le…","""Paolo Bonzini_pbonzini@redhat.…"


In [24]:
for i, row in enumerate(res.iter_rows(named=True)):
    print(f'###### version {i} ##########')
    print(row['func_body'])
    
    

###### version 0 ##########
{
    uint32_t romask = ldl_le_p(&s->romask[addr]);
    uint32_t w1cmask = ldl_le_p(&s->w1cmask[addr]);
    uint32_t oldval = ldl_le_p(&s->mmior[addr]);
    stl_le_p(&s->mmior[addr],
            ((oldval & romask) | (val & ~romask)) & ~(val & w1cmask));
}
###### version 1 ##########
{
    uint32_t romask = ldl_le_p(&s->romask[addr]);
    uint32_t w1cmask = ldl_le_p(&s->w1cmask[addr]);
    uint32_t oldval = ldl_le_p(&s->mmior[addr]);
    stl_le_p(&s->mmior[addr],
            ((oldval & romask) | (val & ~romask)) & ~(val & w1cmask));
}
###### version 2 ##########
{
    uint32_t romask = ldl_le_p(&s->romask[addr]);
    uint32_t w1cmask = ldl_le_p(&s->w1cmask[addr]);
    uint32_t oldval = ldl_le_p(&s->mmior[addr]);
    stl_le_p(&s->mmior[addr],
            ((oldval & romask) | (val & ~romask)) & ~(val & w1cmask));
}
###### version 3 ##########
{
    uint32_t romask = ldl_le_p(&s->romask[addr]);
    uint32_t w1cmask = ldl_le_p(&s->w1cmask[addr]);
    uint32_t old

In [7]:
print(list(res['func_call'])[1])
print(list(res['func_body'])[1])

amdvi_writel(AMDVIState *s, hwaddr addr, uint32_t val)
{
    uint32_t romask = ldl_le_p(&s->romask[addr]);
    uint32_t w1cmask = ldl_le_p(&s->w1cmask[addr]);
    uint32_t oldval = ldl_le_p(&s->mmior[addr]);
    stl_le_p(&s->mmior[addr],
            ((oldval & romask) | (val & ~romask)) & ~(val & w1cmask));
}


In [31]:
print(list(res['func_call'])[30])
print(list(res['func_body'])[30])

amdvi_inval_inttable(AMDVIState *s, uint64_t *cmd)
{
    if (extract64(cmd[0], 16, 44) || cmd[1]) {
        amdvi_log_illegalcom_error(s, extract64(cmd[0], 60, 4),
                                   s->cmdbuf + s->cmdbuf_head);
        return;
    }

    trace_amdvi_intr_inval();
}


# Use LLM to analyze code

In [90]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [91]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load the Qwen-2.5-Coder model and tokenizer
model_name = "Qwen/Qwen2.5-Coder-3B-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True, use_auth_token=True).to(device)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [82]:
def harmonic_mean(values):
    if not values or any(v <= 0 for v in values):
        raise ValueError("Values must be a non-empty list of positive numbers.")
    
    n = len(values)
    reciprocals_sum = sum(1 / v for v in values)
    return n / reciprocals_sum

In [92]:
import timeit

# Function to predict the function name
def predict_function_name(function_text, new_tokens=20):
    # Prepare the prompt for instruction-based inference
    satrt = timeit.default_timer()
    prompt = (    
        f"{function_text}\n\n"
        "Function name:"
    )
    
    # Tokenize the input
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)
    
    # Generate output
    with torch.no_grad():
        outputs = model.generate(
            inputs["input_ids"],
            max_new_tokens=new_tokens,
            do_sample=False,  # Enable sampling for diversity
            temperature=0.1,  # Control creativity
            return_dict_in_generate=True,
            output_scores=True,
        )
    
    print('LLM run time:', timeit.default_timer() - satrt)
    generated_text = tokenizer.decode(outputs.sequences[0], skip_special_tokens=True)
    m = re.search(r"Function name:\s*(\w+)", generated_text)
    predicted_name = m.group(0).replace('Function name:', '')
    last_token = tokenizer('\n', add_special_tokens=False)['input_ids'][0]
    probs = []
    for i, tok_id in enumerate(outputs.sequences[0, -new_tokens:]):
        if tok_id != last_token:
            probabilities = torch.softmax(outputs.scores[i], dim=-1)
            probability = probabilities[0, tok_id].item()
            print(f'Token: {tokenizer.decode(tok_id)} ({tok_id}), Probability: {probability}')
            probs.append(probability)
        else:
            break
        
    
    
    
    return predicted_name, harmonic_mean(probs), outputs

# Example usage
function_text = """
{
    uint32_t romask = ldl_le_p(&s->romask[addr]);
    uint32_t w1cmask = ldl_le_p(&s->w1cmask[addr]);
    uint32_t oldval = ldl_le_p(&s->mmior[addr]);
    stl_le_p(&s->mmior[addr],
            ((oldval & romask) | (val & ~romask)) & ~(val & w1cmask));
}
"""
predicted_name, probability, outputs = predict_function_name(function_text)
print(f"Predicted Function Name: {predicted_name}")
print(f"Probability: {probability:.4f}")


/home/maorreu/.conda/envs/stable_diffusion2/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/maorreu/.conda/envs/stable_diffusion2/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


LLM run time: 2.176042143255472
Token:  handle (3705), Probability: 0.08381908386945724
Token: _mm (35599), Probability: 0.11036155372858047
Token: io (815), Probability: 0.8903297781944275
Token: _write (9165), Probability: 0.42560330033302307
Token: _ (62), Probability: 0.21752646565437317
Token: 3 (18), Probability: 0.2875877618789673
Token: 2 (17), Probability: 0.9039826393127441
Token: 

 (271), Probability: 0.3522231876850128
Token: Description (5009), Probability: 0.33610352873802185
Token: : (25), Probability: 0.6322996616363525
Token:  This (1096), Probability: 0.509026288986206
Token:  function (729), Probability: 0.987256646156311
Token:  handles (13469), Probability: 0.7954902052879333
Token:  the (279), Probability: 0.3898567259311676
Token:  write (3270), Probability: 0.7163492441177368
Token:  operation (5666), Probability: 0.8368138670921326
Token:  to (311), Probability: 0.6931502223014832
Token:  a (264), Probability: 0.539318323135376
Token:   (220), Probability: 0.7

In [93]:
tokenizer.decode(outputs.sequences[0, -20:])

' handle_mmio_write_32\n\nDescription: This function handles the write operation to a 3'

In [94]:
outputs.sequences[0, -20:]

tensor([ 3705, 35599,   815,  9165,    62,    18,    17,   271,  5009,    25,
         1096,   729, 13469,   279,  3270,  5666,   311,   264,   220,    18],
       device='cuda:0')

In [96]:
tokenizer('\n\n', add_special_tokens=False)['input_ids'][0]

271

In [42]:
# lazy_df = lazy_df.join(
#     lazy_df.group_by("func_id").agg(
#         pl.col("commit").shift(1).alias("prev_commit")
#     ).explode("prev_commit"),
#     on=["func_id"],
#     how="left"
# )

In [19]:
pd.read_parquet('df.parquet.gzip') 

,entry_id,commit,func_id,timestamp,func_call,func_name,func_param,func_body,prev_commit
0,3b744ccd4a721e8215e53df3753dfac50199fe55,Peter Maydell_peter.maydell@linaro.org_1732455546,print_ipc_cmd(int cmd),1970-01-01 00:00:01.732455546,print_ipc_cmd(int cmd),print_ipc_cmd,(int cmd),{\n#define output_cmd(val) \\nif( cmd == val )...,None
1,3b744ccd4a721e8215e53df3753dfac50199fe55,Peter Maydell_peter.maydell@linaro.org_1732455546,print_signal_1(abi_ulong arg),1970-01-01 00:00:01.732455546,print_signal_1(abi_ulong arg),print_signal_1,(abi_ulong arg),{\n if (arg < ARRAY_SIZE(target_signal_name...,None
2,3b744ccd4a721e8215e53df3753dfac50199fe55,Peter Maydell_peter.maydell@linaro.org_1732455546,"print_signal(abi_ulong arg, int last)",1970-01-01 00:00:01.732455546,"print_signal(abi_ulong arg, int last)",print_signal,"(abi_ulong arg, int last)","{\n print_signal_1(arg);\n qemu_log(""%s""...",None
3,3b744ccd4a721e8215e53df3753dfac50199fe55,Peter Maydell_peter.maydell@linaro.org_1732455546,print_si_code(int arg),1970-01-01 00:00:01.732455546,print_si_code(int arg),print_si_code,(int arg),{\n const char *codename = NULL;\n\n swi...,None
4,3b744ccd4a721e8215e53df3753dfac50199fe55,Peter Maydell_peter.maydell@linaro.org_1732455546,"get_target_siginfo(target_siginfo_t *tinfo,\n ...",1970-01-01 00:00:01.732455546,"get_target_siginfo(target_siginfo_t *tinfo,\n ...",get_target_siginfo,"(target_siginfo_t *tinfo,\n ...",{\n abi_ulong sival_ptr;\n\n int sig;\n ...,None
...,...,...,...,...,...,...,...,...,...
5618,8527bce6eeffa95a7a9f91325cf5d120abb7d51e,Paolo Bonzini_pbonzini@redhat.com_1731137647,hvf_arch_remove_all_hw_breakpoints(void),1970-01-01 00:00:01.731137647,hvf_arch_remove_all_hw_breakpoints(void),hvf_arch_remove_all_hw_breakpoints,(void),{\n},Paolo Bonzini_pbonzini@redhat.com_1731137647
5619,8527bce6eeffa95a7a9f91325cf5d120abb7d51e,Paolo Bonzini_pbonzini@redhat.com_1731137647,hvf_arch_update_guest_debug(CPUState *cpu),1970-01-01 00:00:01.731137647,hvf_arch_update_guest_debug(CPUState *cpu),hvf_arch_update_guest_debug,(CPUState *cpu),{\n},Paolo Bonzini_pbonzini@redhat.com_1731137647
5620,8527bce6eeffa95a7a9f91325cf5d120abb7d51e,Paolo Bonzini_pbonzini@redhat.com_1731137647,hvf_arch_supports_guest_debug(void),1970-01-01 00:00:01.731137647,hvf_arch_supports_guest_debug(void),hvf_arch_supports_guest_debug,(void),{\n return false;\n},Paolo Bonzini_pbonzini@redhat.com_1731137647
5621,af9ee17a111d6d1dfd02dae38d2917b0d8b54584,Paolo Bonzini_pbonzini@redhat.com_1731137647,cache_host_xcr0(void),1970-01-01 00:00:01.731137647,cache_host_xcr0(void),cache_host_xcr0,(void),{\n if (cached_xcr0) {\n return;\n ...,Peter Maydell_peter.maydell@linaro.org_1731155641


In [13]:
df['timestamp'] = pd.to_datetime(df['timestamp'])

In [125]:
func['func_call']

'replication_stop_all(bool failover, Error **errp)'

In [99]:
df.groupby('func_id')['commit'].shift(1)

0                                                 NaN
1                                                 NaN
2                                                 NaN
3                                                 NaN
4                                                 NaN
                             ...                     
78700    Paolo Bonzini_pbonzini@redhat.com_1731137647
78701    Paolo Bonzini_pbonzini@redhat.com_1731137647
78702    Paolo Bonzini_pbonzini@redhat.com_1731137647
78703    Paolo Bonzini_pbonzini@redhat.com_1731137647
78704    Paolo Bonzini_pbonzini@redhat.com_1731137647
Name: commit, Length: 78705, dtype: object

In [107]:
commit_count = sum(1 for commit in repo.walk(repo.head.target, pygit2.GIT_SORT_TOPOLOGICAL))
for commit in tqdm(repo.walk(repo.head.target, pygit2.GIT_SORT_TOPOLOGICAL), total=commit_count, desc='process commits'):

    print(len([entry for entry in commit.tree if entry.name.endswith('.c')]))

process commits:   0%|          | 0/117067 [00:00<?, ?it/s]

24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
24
2

41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
41
4

KeyboardInterrupt: 

In [120]:
commit.__str__()

'<pygit2.Object{commit:749ad5e887e85fd51d83bf4d08ed72858f937fd9}>'

In [121]:
repo.diff('commit:749ad5e887e85fd51d83bf4d08ed72858f937fd9', 'commit:749ad5e887e85fd51d83bf4d08ed72858f937fd9')

KeyError: 'commit:749ad5e887e85fd51d83bf4d08ed72858f937fd9'

In [104]:
print(commit.message)

Merge tag 'pull-lu-20241122' of https://gitlab.com/rth7680/qemu into staging

linux-user: Fix strace output for s390x mmap()
linux-user: Print tid not pid with strace

# -----BEGIN PGP SIGNATURE-----
#
# iQFRBAABCgA7FiEEekgeeIaLTbaoWgXAZN846K9+IV8FAmdA59IdHHJpY2hhcmQu
# aGVuZGVyc29uQGxpbmFyby5vcmcACgkQZN846K9+IV+YfAf9E41MJqZhhtWIHJW7
# z+mDw3Yr4Lgvmu2q3PmvOfM9YfvT3HFCh4gQRDaZbd3zy3c73S1/xSwBF2MQhElf
# Ml5mEUXtZGrK6Z/hq0QfS2mOGdwsydy+OpEtaw7YhVCMxMapnMi8rvSKe5XRpy/2
# vragLVvJxISS6TnnJ/5RcnmajRLzgGIj7Ej4lbCyViOKhDPXMCr77+jOPHSxUt4R
# PQMPzUPJG8Y7j+YR4fQng6700XaqIu8hcn7o0RSghdWEvF+fPaJZDlF32V+yD2Jj
# sAvtRsiwLM54rycMJUrQ274IYPIhxyxlgWT0mIIb9fXYdYTHcpZ+ggQ/mSKg+8QQ
# 9tmsRg==
# =pR93
# -----END PGP SIGNATURE-----
# gpg: Signature made Fri 22 Nov 2024 20:21:38 GMT
# gpg:                using RSA key 7A481E78868B4DB6A85A05C064DF38E8AF7E215F
# gpg:                issuer "richard.henderson@linaro.org"
# gpg: Good signature from "Richard Henderson <richard.henderson@linaro.org>" [full]
# Prima